In [48]:
import numpy as np
import pandas as pd
import sklearn as sk
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

Data Exploration and Preprocessing


In [49]:
data = pd.read_csv('data/AABA_2006-01-01_to_2018-01-01.csv')
data.describe()

,Open,High,Low,Close,Volume
count,3019.000000,3019.000000,3019.000000,3019.000000,3.019000e+03
mean,28.426479,28.766532,28.066558,28.412726,2.158391e+07
std,13.257242,13.356692,13.157326,13.258163,1.926231e+07
min,9.100000,9.480000,8.940000,8.950000,1.939061e+06
25%,16.175000,16.385000,15.970000,16.130000,1.248025e+07
50%,27.180000,27.490000,26.820000,27.100000,1.732130e+07
75%,36.655000,37.035000,36.305000,36.635000,2.512757e+07
max,73.020000,73.250000,72.460000,72.930000,4.382317e+08


In [50]:
index_fund_data = pd.read_csv('data/SPX.csv')
index_fund_data = index_fund_data[index_fund_data['Date'] > '2010-01-01']
index_fund_data['Date'] = pd.to_datetime(index_fund_data['Date'])
index_fund_data.drop(columns=['Adj Close'], inplace=True)
data = data.bfill()

In [51]:
data.head(5)

,Date,Open,High,Low,Close,Volume,Name
0,2006-01-03,39.69,41.22,38.79,40.91,24232729,AABA
1,2006-01-04,41.22,41.90,40.77,40.97,20553479,AABA
2,2006-01-05,40.93,41.73,40.85,41.53,12829610,AABA
3,2006-01-06,42.88,43.57,42.80,43.21,29422828,AABA
4,2006-01-09,43.10,43.66,42.82,43.42,16268338,AABA


In [52]:
data = data[data['Date'] > '2010-01-01']
data = data.reset_index(drop=True)
data.head(5)

,Date,Open,High,Low,Close,Volume,Name
0,2010-01-04,16.94,17.20,16.88,17.10,16588957,AABA
1,2010-01-05,17.22,17.23,17.00,17.23,11718126,AABA
2,2010-01-06,17.17,17.30,17.07,17.17,16421960,AABA
3,2010-01-07,16.81,16.90,16.57,16.70,31816301,AABA
4,2010-01-08,16.68,16.76,16.62,16.70,15471074,AABA


In [53]:
data['Date'] = pd.to_datetime(data['Date'])
data = data.bfill()
data.describe()

,Date,Open,High,Low,Close,Volume
count,2012,2012.000000,2012.000000,2012.000000,2012.00000,2.012000e+03
mean,2014-01-01 16:56:18.131212,30.941456,31.277594,30.599210,30.93997,1.888270e+07
min,2010-01-04 00:00:00,11.300000,11.800000,11.090000,11.09000,1.939061e+06
25%,2012-01-02 00:00:00,16.190000,16.400000,16.000000,16.14750,1.097719e+07
50%,2014-01-02 12:00:00,32.050000,32.560000,31.560000,32.13500,1.545990e+07
75%,2016-01-01 00:00:00,40.910000,41.272500,40.425000,40.93000,2.218065e+07
max,2017-12-29 00:00:00,73.020000,73.250000,72.460000,72.93000,2.693771e+08
std,NaN,14.818074,14.945141,14.689174,14.81872,1.523157e+07


In [54]:
# Drop old index columns if they exist
for col in ['index_open', 'index_high', 'index_low', 'index_close', 'index_volume']:
    if col in data.columns:
        data = data.drop(columns=col)

# Rename and merge
index_renamed = index_fund_data.rename(
    columns={
        'Open': 'index_open',
        'High': 'index_high',
        'Low': 'index_low',
        'Close': 'index_close',
        'Volume': 'index_volume'
    }
)
data = data.merge(index_renamed, on='Date', how='left')
data.head()

,Date,Open,High,Low,Close,Volume,Name,index_open,index_high,index_low,index_close,index_volume
0,2010-01-04,16.94,17.20,16.88,17.10,16588957,AABA,1116.560059,1133.869995,1116.560059,1132.989990,3991400000
1,2010-01-05,17.22,17.23,17.00,17.23,11718126,AABA,1132.660034,1136.630005,1129.660034,1136.520020,2491020000
2,2010-01-06,17.17,17.30,17.07,17.17,16421960,AABA,1135.709961,1139.189941,1133.949951,1137.140015,4972660000
3,2010-01-07,16.81,16.90,16.57,16.70,31816301,AABA,1136.270020,1142.459961,1131.319946,1141.689941,5270680000
4,2010-01-08,16.68,16.76,16.62,16.70,15471074,AABA,1140.520020,1145.390015,1136.219971,1144.979980,4389590000


In [55]:
data.shape

(2012, 12)

Split into Train, Test, and Validation Sets

In [56]:
cv_splits = TimeSeriesSplit(n_splits=5)

In [57]:
iso_param_grid = {
    'n_estimators': [50, 100, 200, 500],
    'max_features': [0.5, 0.8, 1.0],
    'contamination': [0.01, 0.03, 0.05, 0.1]
}

rf_param_grid = {
    'n_estimators': [50, 100, 200, 500],
    'max_features': ['sqrt', 'log2']
}

Feature Engineering

In [58]:
# Cross Validation Loop
cv_iso_train_reports = []
cv_iso_test_reports = []

cv_rf_train_reports = []
cv_rf_test_reports = []

for train_index, test_index in cv_splits.split(data):
    X_train = data.iloc[train_index].copy()
    X_test = data.iloc[test_index].copy()

    # --- Feature Engineering with .loc ---

    #Log return shows the percentage change in price, which is a common feature in financial time series analysis. 
    # It helps to capture the relative change in price rather than the absolute change, making it easier to compare across different time periods and assets.
    X_train.loc[:, 'log_return'] = np.log(X_train['Close'] / X_train['Close'].shift(1))
    X_test.loc[:, 'log_return'] = np.log(X_test['Close'] / X_test['Close'].shift(1))

    X_train.loc[:, 'excess_return'] = X_train['log_return'] - np.log(X_train['index_close'] / X_train['index_close'].shift(1))
    X_test.loc[:, 'excess_return'] = X_test['log_return'] - np.log(X_test['index_close'] / X_test['index_close'].shift(1))

    mean_excess = X_train['excess_return'].mean()
    std_excess = X_train['excess_return'].std()

    X_train.loc[:, 'excess_z'] = (X_train['excess_return'] - mean_excess) / std_excess
    X_test.loc[:, 'excess_z'] = (X_test['excess_return'] - mean_excess) / std_excess
    threshold = 3

    y_train = (X_train['excess_z'].abs() > threshold).astype(int)      
    y_test = (X_test['excess_z'].abs() > threshold).astype(int)

    # These are the rolling standard deviations of the log returns, which are commonly used as measures of volatility in financial time series.
    X_train.loc[:, 'vol_5'] = X_train['log_return'].rolling(5).std()
    X_train.loc[:, 'vol_20'] = X_train['log_return'].rolling(20).std()
    X_train.loc[:, 'vol_60'] = X_train['log_return'].rolling(60).std()

    X_test.loc[:, 'vol_5'] = X_test['log_return'].rolling(5).std()
    X_test.loc[:, 'vol_20'] = X_test['log_return'].rolling(20).std()
    X_test.loc[:, 'vol_60'] = X_test['log_return'].rolling(60).std()

    # The range volatility is a measure of the price range relative to the closing price, which can indicate the level of price fluctuation within a trading period.
    X_train.loc[:, 'range_vol'] = (X_train['High'] - X_train['Low']) / X_train['Close']
    X_test.loc[:, 'range_vol'] = (X_test['High'] - X_test['Low']) / X_test['Close']

    # Volume features
    X_train.loc[:, 'vol_mean'] = X_train['Volume'].rolling(20).mean()
    X_train.loc[:, 'vol_std'] = X_train['Volume'].rolling(20).std()
    X_train.loc[:, 'volume_z'] = (X_train['Volume'] - X_train['vol_mean']) / X_train['vol_std']

    X_test.loc[:, 'vol_mean'] = X_test['Volume'].rolling(20).mean()
    X_test.loc[:, 'vol_std'] = X_test['Volume'].rolling(20).std()
    X_test.loc[:, 'volume_z'] = (X_test['Volume'] - X_test['vol_mean']) / X_test['vol_std']

    # Drop NaNs caused by rolling windows, not sure if this is a great idea...
    X_train = X_train.dropna()
    X_test = X_test.dropna()

    # Align y_train/y_test
    y_train = y_train.loc[X_train.index]
    y_test = y_test.loc[X_test.index]

    # Select Features
    features = ['log_return', 'vol_5', 'vol_20', 'vol_60', 'range_vol', 'vol_mean', 'vol_std', 'volume_z']
    X_train =  pd.DataFrame(X_train, columns=features)
    X_test = pd.DataFrame(X_test, columns=features)

    # Scale Features
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # convert scaled features back to dataframe to look at metrics
    X_train_scaled =  pd.DataFrame(X_train_scaled, columns=features)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=features)

    # Grid search Isolation Forest
    best_iso_score = -np.inf
    best_iso_model = None
    for params in ParameterGrid(iso_param_grid):
        model = IsolationForest(
            random_state=42,
            **params
        )
        model.fit(X_train_scaled)
        pred_train = (model.predict(X_train_scaled) == -1).astype(int)
        score = recall_score(y_train, pred_train)
        if score > best_iso_score:
            best_iso_score = score
            best_iso_model = model
    print(best_iso_model.get_params())

    pred_iso_train = (best_iso_model.predict(X_train_scaled) == -1).astype(int)
    pred_iso_test = (best_iso_model.predict(X_test_scaled) == -1).astype(int)
    
    cv_iso_train_reports.append(recall_score(y_train, pred_iso_train))
    cv_iso_test_reports.append(recall_score(y_test, pred_iso_test))

    # Grid search Random Forest
    best_rf_score = -np.inf
    best_rf_model = None
    for params in ParameterGrid(rf_param_grid):
        model = RandomForestClassifier(
            random_state=42,
            **params
        )
        model.fit(X_train_scaled, y_train)
        pred_train = model.predict(X_train_scaled)
        score = recall_score(y_train, pred_train)
        if score > best_rf_score:
            best_rf_score = score
            best_rf_model = model
    print(best_rf_model.get_params())

    pred_rf_train = best_rf_model.predict(X_train_scaled)
    pred_rf_test = best_rf_model.predict(X_test_scaled) 

    cv_rf_train_reports.append(recall_score(y_train, pred_rf_train))
    cv_rf_test_reports.append(recall_score(y_test, pred_rf_test))


{'bootstrap': False, 'contamination': 0.03, 'max_features': 0.5, 'max_samples': 'auto', 'n_estimators': 50, 'n_jobs': None, 'random_state': 42, 'verbose': 0, 'warm_start': False}
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 50, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
{'bootstrap': False, 'contamination': 0.1, 'max_features': 0.5, 'max_samples': 'auto', 'n_estimators': 100, 'n_jobs': None, 'random_state': 42, 'verbose': 0, 'warm_start': False}
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_sa

In [59]:
print("ISO CV TRAIN RECALL SCORES:")
print(cv_iso_train_reports)
print("ISO CV TEST RECALL SCORES:")
print(cv_iso_test_reports)

print("RF CV TRAIN RECALL SCORES:")
print(cv_rf_train_reports)
print("RF CV TEST RECALL SCORES:")
print(cv_rf_test_reports)

ISO CV TRAIN RECALL SCORES:
[1.0, 1.0, 1.0, 0.9545454545454546, 1.0]
ISO CV TEST RECALL SCORES:
[0.9090909090909091, 1.0, 0.7777777777777778, 1.0, 1.0]
RF CV TRAIN RECALL SCORES:
[1.0, 1.0, 1.0, 1.0, 1.0]
RF CV TEST RECALL SCORES:
[0.18181818181818182, 1.0, 0.4444444444444444, 0.4, 1.0]
